In [ ]:
import sys
sys.path.append("../")
import utils

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter('ignore')

# Parameters (change as needed)
csv_path = "../data/dataset_1.csv"
seed = 0
n_init = 10
n_steps = 50
shuffle_seed = 0
model_type = 'rf'

# Load data
df_ = pd.read_csv(csv_path)

# Use last column as target (prop)
prop = list(df_.columns)[-1]

# Standardize explanatory features (all columns except the last)
df_.iloc[:, :-1] = StandardScaler().fit_transform(df_.iloc[:, :-1])

# Split target and features
df_oracle = df_[[prop]]
df_desc = df_.drop(prop, axis=1)

# Initial labeled sample
idx_sample = list(df_oracle.sample(n_init, random_state=seed).index)
df = df_oracle.iloc[idx_sample].reset_index(drop=True)
df_X = df_desc.iloc[idx_sample].reset_index(drop=True)

# Pool (remaining)
df_pool = df_oracle.drop(idx_sample).reset_index(drop=True)
df_pool_X = df_desc.drop(idx_sample).reset_index(drop=True)

# Shuffle pool
idx = list(df_pool_X.sample(frac=1, random_state=shuffle_seed).index)
df_pool = df_pool.iloc[idx].reset_index(drop=True)
df_pool_X = df_pool_X.iloc[idx].reset_index(drop=True)

# Records
obs_max = [df[prop].max()]
ec = []

for num in range(n_steps):
    # alpha schedule
    alpha = 1.0 / (num + 1) ** 0.5

    # Get conformal quantile predictions for pool
    y_pred, y_q = utils.weighted_conformal_quantile(df_X, df[prop], df_pool_X, alpha=alpha, model_type=model_type)

    y_pred_ub = y_pred + y_q
    y_pred_lb = y_pred - y_q

    # Coverage on the pool
    ec.append(np.sum((df_pool[prop].values >= y_pred_lb) & (df_pool[prop].values <= y_pred_ub)) / len(df_pool))

    # Acquisition: uncertainty
    range_val = df[prop].max() - df[prop].min()
    current_max = df[prop].max()
    acquisition_function = np.array([
        utils.sigma(lb, ub, range_val, alpha)
        for lb, ub in zip(y_pred_lb, y_pred_ub)
    ])

    # Select top-1 (simpler argmax)
    new_index = [int(np.argmax(acquisition_function))]

    # Move selected sample from pool to labeled set
    df_X = pd.concat([df_X, df_pool_X.iloc[new_index]]).reset_index(drop=True)
    df_pool_X = df_pool_X.drop(new_index).reset_index(drop=True)

    df = pd.concat([df, df_pool.iloc[new_index]]).reset_index(drop=True)
    df_pool = df_pool.drop(new_index).reset_index(drop=True)
